# AI Adoption in UK Small and Medium Businesses

**Analysis of ONS Business Insights and Conditions Survey (BICS) Data**

Independent personal research project, unpaid and non-commercial, done in my own time using public data only. Not affiliated with, funded by or representing my employer or any organisation. Not advice.

Data Source: ONS BICS Wave 105 (18-31 March 2024)  
Licence: Open Government Licence v3.0

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

sns.set_style('whitegrid')
sns.set_palette('husl')
%matplotlib inline

## 1. Load Data

In [ ]:
# Load processed datasets
data_dir = Path('../data/processed')

current_use = pd.read_csv(data_dir / 'ai_current_use.csv')
purposes = pd.read_csv(data_dir / 'ai_purposes.csv')
methods = pd.read_csv(data_dir / 'ai_adoption_methods.csv')
planned = pd.read_csv(data_dir / 'ai_planned_adoption.csv')
barriers = pd.read_csv(data_dir / 'ai_barriers.csv')

print(f"Current AI use: {len(current_use)} records")
print(f"AI purposes: {len(purposes)} records")
print(f"Adoption methods: {len(methods)} records")
print(f"Planned adoption: {len(planned)} records")
print(f"Barriers: {len(barriers)} records")

## 2. Research Question

**What drives AI adoption in UK small and medium businesses?**

### Hypotheses:
1. H1: Larger firms have higher AI adoption rates than smaller firms
2. H2: The primary barrier to adoption is lack of expertise, not cost
3. H3: Operational efficiency is the main driver of AI adoption

## 3. Descriptive Statistics

In [ ]:
# AI adoption by firm size
size_order = ['Micro (0-9)', 'Small (10-49)', 'Medium (50-249)', 'Large (250+)']

size_data = current_use[current_use['size_band'].isin(size_order)]
size_stats = size_data.groupby('size_band')['adoption_rate'].agg([
    ('mean', 'mean'),
    ('median', 'median'),
    ('std', 'std'),
    ('count', 'count')
]).reset_index()

size_stats['size_band'] = pd.Categorical(size_stats['size_band'], categories=size_order, ordered=True)
size_stats = size_stats.sort_values('size_band')

print("AI Adoption by Firm Size:")
print(size_stats.to_string(index=False))
print(f"\nAll figures calculated from ONS BICS Wave 105 (March 2024)")

In [ ]:
# Visualise adoption by size
fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(range(len(size_stats)), size_stats['mean'], 
               yerr=size_stats['std'], capsize=5,
               color=['#3498db', '#2ecc71', '#f39c12', '#e74c3c'])

ax.set_xticks(range(len(size_stats)))
ax.set_xticklabels(size_stats['size_band'], rotation=15, ha='right')
ax.set_ylabel('Mean AI Adoption Rate', fontsize=12)
ax.set_xlabel('Firm Size', fontsize=12)
ax.set_title('AI Adoption by Firm Size\n(ONS BICS Wave 105, March 2024)', 
             fontsize=14, fontweight='bold')

for i, (bar, val) in enumerate(zip(bars, size_stats['mean'])):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.1%}',
            ha='center', va='bottom', fontsize=10)

ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
plt.tight_layout()
plt.show()

In [ ]:
# Top barriers to AI adoption
barriers_agg = barriers.groupby('barrier')['percentage'].mean().sort_values(ascending=False)

print("Top Barriers to AI Adoption:")
for i, (barrier, pct) in enumerate(barriers_agg.head(8).items(), 1):
    print(f"{i}. {barrier[:70]:70s} {pct:6.1%}")

print(f"\nCalculated from ONS BICS Wave 105 (March 2024)")

In [ ]:
# Top purposes for AI use
purposes_agg = purposes.groupby('purpose')['percentage'].mean().sort_values(ascending=False)

print("What Businesses Use AI For:")
for i, (purpose, pct) in enumerate(purposes_agg.head(8).items(), 1):
    print(f"{i}. {purpose[:70]:70s} {pct:6.1%}")

print(f"\nCalculated from ONS BICS Wave 105 (March 2024)")

## 4. Statistical Analysis

In [ ]:
from scipy import stats

# Test correlation between firm size and AI adoption
size_data_test = current_use[current_use['size_band'].isin(size_order)].copy()
size_data_test['size_numeric'] = size_data_test['size_band'].map({
    'Micro (0-9)': 1,
    'Small (10-49)': 2,
    'Medium (50-249)': 3,
    'Large (250+)': 4
})

corr, p_value = stats.spearmanr(size_data_test['size_numeric'], 
                                 size_data_test['adoption_rate'])

print(f"Spearman correlation (firm size vs AI adoption): {corr:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"Interpretation: {'Significant' if p_value < 0.05 else 'Not significant'} at α=0.05")
print(f"\nH1 (Larger firms have higher AI adoption): {'Supported' if p_value < 0.05 and corr > 0 else 'Not supported'}")

In [ ]:
# Test H2: Primary barrier is expertise, not cost
# Extract only the substantive barriers (exclude "not attempted" and "not prevented")
substantive_barriers = barriers[
    ~barriers['barrier'].str.contains('not attempted|not been prevented', case=False, na=False)
]

barriers_mean = substantive_barriers.groupby('barrier')['percentage'].mean().sort_values(ascending=False)

top_barrier = barriers_mean.index[0]
top_barrier_pct = barriers_mean.iloc[0]

print(f"Top substantive barrier: {top_barrier}")
print(f"Percentage: {top_barrier_pct:.1%}")

expertise_barriers = barriers_mean[barriers_mean.index.str.contains('expertise', case=False)]
cost_barriers = barriers_mean[barriers_mean.index.str.contains('cost', case=False)]

if len(expertise_barriers) > 0 and len(cost_barriers) > 0:
    print(f"\nExpertise barrier: {expertise_barriers.iloc[0]:.1%}")
    print(f"Cost barrier: {cost_barriers.iloc[0]:.1%}")
    print(f"\nH2 (Expertise > Cost as barrier): {'Supported' if expertise_barriers.iloc[0] > cost_barriers.iloc[0] else 'Not supported'}")

In [ ]:
# Test H3: Operational efficiency is main driver
purposes_excl = purposes[
    ~purposes['purpose'].str.contains('Not sure|Not applicable', case=False, na=False)
]

purposes_mean = purposes_excl.groupby('purpose')['percentage'].mean().sort_values(ascending=False)

top_purpose = purposes_mean.index[0]
top_purpose_pct = purposes_mean.iloc[0]

print(f"Top purpose for AI use: {top_purpose}")
print(f"Percentage: {top_purpose_pct:.1%}")

operations_purposes = purposes_mean[purposes_mean.index.str.contains('operation', case=False)]

if len(operations_purposes) > 0:
    print(f"\nOperations improvement: {operations_purposes.iloc[0]:.1%}")
    print(f"\nH3 (Operations is main driver): {'Supported' if 'operation' in top_purpose.lower() else 'Partially supported'}")

## 5. Results Summary

All figures in this notebook are calculated from ONS BICS Wave 105 data (18-31 March 2024), licensed under the Open Government Licence v3.0.

**Data source URL:**  
https://www.ons.gov.uk/economy/economicoutputandproductivity/output/datasets/businessinsightsandimpactontheukeconomy

**Licence:**  
Open Government Licence v3.0  
https://www.nationalarchives.gov.uk/doc/open-government-licence/version/3/